# 02 Feature Matrix Inspection

Repository notebook for inspecting SACU feature matrices, pathway-specific feature groups, missingness, scaling, and split alignment.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "src").exists():
            return candidate
    return Path.cwd()

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")

def read_optional_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing file: {path}")
        return pd.DataFrame()
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() == ".tsv":
        return pd.read_csv(path, sep="\t")
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported table format: {path.suffix}")

def find_files(patterns, roots):
    rows = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for pattern in patterns:
            for path in root.rglob(pattern):
                rows.append({
                    "file": path.name,
                    "path": str(path.relative_to(PROJECT_ROOT)) if PROJECT_ROOT in path.parents else str(path),
                    "size_kb": round(path.stat().st_size / 1024, 2),
                    "modified": pd.to_datetime(path.stat().st_mtime, unit="s"),
                })
    return pd.DataFrame(rows).sort_values("modified", ascending=False).reset_index(drop=True) if rows else pd.DataFrame()

## Locate feature matrices

In [ ]:
feature_files = find_files(
    patterns=["*feature*.csv", "*matrix*.csv", "*X_*.csv", "*modeling*.csv"],
    roots=[PROJECT_ROOT / "outputs", PROJECT_ROOT / "data", PROJECT_ROOT / "results"],
)
feature_files

## Load feature matrix

In [ ]:
FEATURE_PATH = None

if FEATURE_PATH is None and not feature_files.empty:
    FEATURE_PATH = PROJECT_ROOT / feature_files.iloc[0]["path"]

feature_df = read_optional_table(Path(FEATURE_PATH)) if FEATURE_PATH is not None else pd.DataFrame()
print(f"Loaded feature matrix shape: {feature_df.shape}")
feature_df.head()

## Numeric feature audit

In [ ]:
if feature_df.empty:
    numeric_audit = pd.DataFrame()
else:
    numeric_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_audit = pd.DataFrame([
        {"metric": "n_rows", "value": len(feature_df)},
        {"metric": "n_columns", "value": feature_df.shape[1]},
        {"metric": "n_numeric_columns", "value": len(numeric_cols)},
        {"metric": "missing_numeric_values", "value": int(feature_df[numeric_cols].isna().sum().sum()) if numeric_cols else 0},
        {"metric": "infinite_numeric_values", "value": int(np.isinf(feature_df[numeric_cols].to_numpy()).sum()) if numeric_cols else 0},
    ])
numeric_audit

## Pathway feature grouping

In [ ]:
pathway_patterns = {
    "local_regional": ["local", "regional", "left_cc", "left_mlo", "right_cc", "right_mlo"],
    "multiview": ["multiview", "cc_mlo", "view_pair"],
    "bilateral": ["bilateral", "asymmetry", "left_right"],
    "temporal_spatial": ["temporal", "spatial", "progression", "prior", "gap"],
    "metadata": ["metadata", "age", "density", "manufacturer", "laterality"],
    "adaptive_control": ["mask", "availability", "completeness", "active_pathway", "complexity"],
}

feature_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist() if not feature_df.empty else []
rows = []

for pathway, tokens in pathway_patterns.items():
    matched = [
        col for col in feature_cols
        if any(token in col.lower() for token in tokens)
    ]
    rows.append({
        "pathway": pathway,
        "n_features": len(matched),
        "example_features": ", ".join(matched[:5]),
    })

pathway_feature_summary = pd.DataFrame(rows)
pathway_feature_summary

## Missingness and variance inspection

In [ ]:
if not feature_df.empty and feature_cols:
    feature_quality = pd.DataFrame({
        "feature": feature_cols,
        "missing_fraction": feature_df[feature_cols].isna().mean().values,
        "std": feature_df[feature_cols].std(numeric_only=True).values,
        "min": feature_df[feature_cols].min(numeric_only=True).values,
        "max": feature_df[feature_cols].max(numeric_only=True).values,
    })
    feature_quality["near_zero_variance"] = feature_quality["std"].fillna(0) < 1e-12
    feature_quality = feature_quality.sort_values(["missing_fraction", "near_zero_variance"], ascending=False)
else:
    feature_quality = pd.DataFrame()

feature_quality.head(30)

## Correlation with target, when available

In [ ]:
target_candidates = ["diagnosis_label", "target", "label", "cancer", "malignant"]
target_col = next((c for c in target_candidates if c in feature_df.columns), None)

if target_col is not None and feature_cols:
    correlations = []
    y = pd.to_numeric(feature_df[target_col], errors="coerce")
    for col in feature_cols:
        x = pd.to_numeric(feature_df[col], errors="coerce")
        if x.notna().sum() > 2 and x.std() > 0:
            correlations.append({"feature": col, "target_correlation": x.corr(y)})
    corr_df = pd.DataFrame(correlations)
    corr_df["abs_target_correlation"] = corr_df["target_correlation"].abs()
    corr_df = corr_df.sort_values("abs_target_correlation", ascending=False)
else:
    corr_df = pd.DataFrame()

corr_df.head(30)